
# pluto control

In [3]:
import io
import threading
import time
import adi
import ipywidgets as widgets
from IPython.display import display
import matplotlib
matplotlib.use("Agg")  # Headless renderer, completely thread-safe
import matplotlib.pyplot as plt
import numpy as np

# 1. Initialize PlutoSDR
sdr = adi.Pluto("ip:192.168.2.1")
try:
    sdr._ctrl.debug_attrs["calib_mode"] = "auto"
except Exception:
    pass

sdr.gain_control_mode_chan0 = "manual"
sdr.sample_rate = int(2e6)
sdr.rx_buffer_size = 1024

# SDR capabilities (AD9363 datasheet: https://www.analog.com/media/en/technical-documentation/data-sheets/ad9363.pdf)
# Frequencies are in MHz
min_tuning_freq = 70.0
max_tuning_freq = 3800.0
min_gain = 0
max_gain = 69 # technicaly 74.5 but the code breaks past this point
min_bandwidth = 0.2
max_bandwidth = 20.0

# 2. UI Controls
slider_layout = widgets.Layout(width='600px')
freq_slider = widgets.FloatSlider(value=433.92, min=min_tuning_freq, max=max_tuning_freq, step=0.1, description='Freq (MHz):', style={'description_width': 'initial'}, layout=slider_layout)
bandwidth_slider = widgets.FloatSlider(value=2.0, min=min_bandwidth, max=max_bandwidth, step=0.1, description='BW (MHz):', style={'description_width': 'initial'}, layout=slider_layout)
gain_slider = widgets.IntSlider(value=40, min=min_gain, max=max_gain, step=1, description='Gain (dB):', style={'description_width': 'initial'}, layout=slider_layout)
update_period = widgets.FloatSlider(value=0.5, min=0.1, max=10.0, step=0.1, description='Period (s):', style={'description_width': 'initial'}, layout=slider_layout)
start_button = widgets.ToggleButton(value=False, description='Start Live Stream', button_style='success', icon='play')

# Thread-safe Image widget instead of Output()
plot_image = widgets.Image(format='png', width=800, height=320)
status_label = widgets.Label(value="Status: Idle")

is_streaming = False
stream_thread = None

def render_plot_bytes(x_data, psd, center_freq, bw_mhz):
    fig, ax = plt.subplots(figsize=(10, 4), dpi=80)
    ax.plot(x_data, psd, color='cyan')
    ax.set_facecolor('#111111')
    ax.set_title(f"PlutoSDR Live Stream @ {center_freq:.2f} MHz (BW: {bw_mhz:.1f} MHz)")
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Relative Power (dB)")
    ax.grid(True, which='both', linestyle='--', alpha=0.4)
    ax.set_ylim(-20, 80)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()

def stream_worker():
    global is_streaming
    while is_streaming:
        try:
            # Apply slider values to hardware
            sdr.rx_lo = int(freq_slider.value * 1e6)
            sdr.rx_rf_bandwidth = int(bandwidth_slider.value * 1e6)
            sdr.rx_hardwaregain_chan0 = int(gain_slider.value)

            # Capture & DSP
            samples = sdr.rx()
            if not is_streaming:
                break

            window = np.hanning(len(samples))
            windowed_samples = samples * window
            fft_data = np.abs(np.fft.fftshift(np.fft.fft(windowed_samples)))**2 / np.sum(window**2)
            psd = 10 * np.log10(fft_data + 1e-12)
            f = np.linspace(-sdr.sample_rate / 2, sdr.sample_rate / 2, len(samples)) / 1e6
            center_freq = freq_slider.value

            # Push PNG bytes directly to the Image widget (thread-safe)
            plot_image.value = render_plot_bytes(center_freq + f, psd, center_freq, bandwidth_slider.value)
            status_label.value = f"Status: Streaming @ {center_freq:.2f} MHz | Gain: {gain_slider.value} dB"

            time.sleep(update_period.value)
        except Exception as e:
            status_label.value = f"Error: {e}"
            break

def on_toggle_change(change):
    global is_streaming, stream_thread
    if change['new']:
        is_streaming = True
        start_button.description = 'Stop Stream'
        start_button.button_style = 'danger'
        start_button.icon = 'stop'
        stream_thread = threading.Thread(target=stream_worker, daemon=True)
        stream_thread.start()
    else:
        is_streaming = False
        start_button.description = 'Start Live Stream'
        start_button.button_style = 'success'
        start_button.icon = 'play'
        status_label.value = "Status: Stopped"

start_button.observe(on_toggle_change, names='value')

# Seed an initial blank plot so the Image widget has valid PNG data immediately
plot_image.value = render_plot_bytes(np.linspace(432.92, 434.92, 1024), np.zeros(1024), 433.92, 2.0)

display(widgets.VBox([
    widgets.HTML("<h3>PlutoSDR Live Control Panel</h3>"),
    freq_slider,
    bandwidth_slider,
    gain_slider,
    update_period,
    widgets.HBox([start_button, status_label]),
    plot_image
]))

# test your sdr is working
run this program if near an air port tune to 125MHz